# Tech Challenge - Análise

**Pergunta Norteadora:** O que levou o setor a entrar em um platô de crescimento a partir de 2018, e quias ações podem retomar a trajetória de crescimento observada em 2017?

In [43]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

print ("Bibliotecas Importadas")

Bibliotecas Importadas


In [44]:
pd.set_option('display.max_columns', None)

DIR = Path('data/clean')

In [45]:
df_combined    = pd.read_parquet(DIR / 'df_combined.parquet')
df_order_items = pd.read_parquet(DIR / 'df_order_items.parquet')
df_orders      = pd.read_parquet(DIR / 'df_orders.parquet')
df_products    = pd.read_parquet(DIR / 'df_products.parquet')

# Grao de item + mes da compra + categoria (usado pelos graficos 2 a 5)
itens = (
    df_order_items
    .merge(df_orders[['order_id', 'ano_mes']], on='order_id', how='inner')
    .merge(df_products[['product_id', 'product_category_name_english', 'product_category_name_pt']],
           on='product_id', how='left')
)

# Paleta categorica fixa (skill de dataviz do projeto) - cor por CATEGORIA, nao por rank
CORES_CATEGORIA = {
    'health_beauty':         '#2a78d6',  # azul
    'watches_gifts':         '#eb6834',  # laranja
    'bed_bath_table':        '#1baf7a',  # verde-agua
    'sports_leisure':        '#eda100',  # amarelo
    'computers_accessories': '#e87ba4',  # magenta
}
TOP5 = list(CORES_CATEGORIA.keys())

print(f'df_combined : {df_combined.shape}')
print(f'itens       : {itens.shape}')

df_combined : (99092, 31)
itens       : (112279, 10)


# 1. Evolução Mensal - Faturamento e Pedidos

Faturamento (`valor_produtos`, preço de tabela - decisão de governança) e números de pedidos, mês a mês, no recorte validado. O platô a partir de 2018-01 fica evidente: depois do pico da black friday (Nov/2017), os meses de 2018 não retoman o ritmo de crescimento de 2017.

In [46]:
mensal = df_combined.groupby('ano_mes').agg(
    faturamento=('valor_produtos', 'sum'),
    pedidos=('order_id','nunique'),
).reset_index()

tabela_mensal = mensal.copy()
tabela_mensal['faturamento'] = tabela_mensal['faturamento'].round(2)
display(tabela_mensal)

,ano_mes,faturamento,pedidos
0,2017-01,120312.87,800
1,2017-02,247303.02,1780
2,2017-03,374344.30,2682
3,2017-04,359927.23,2404
4,2017-05,506071.14,3700
5,2017-06,433038.60,3245
6,2017-07,498031.48,4026
7,2017-08,573971.68,4331
8,2017-09,624401.69,4285
9,2017-10,664219.43,4631


In [47]:
# Grafico Evolução Mensal

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=mensal['ano_mes'], y=mensal['faturamento'], name='Faturamento (R$)',
    mode='lines+markers', line=dict(color='#2a78d6', width=2), marker=dict(size=6),
))
fig.add_trace(go.Scatter(
    x=mensal['ano_mes'], y=mensal['pedidos'], name='Pedidos',
    mode='lines+markers', line=dict(color='#1baf7a', width=2), marker=dict(size=6),
    yaxis='y2',
))
fig.add_vrect(x0='2018-01', x1='2018-08', fillcolor='#e34948', opacity=0.08, line_width=0,
              annotation_text='platô 2018', annotation_position='top left')
fig.update_layout(
    title='Evolução mensal de Faturamento e Pedidos (jan/2017–ago/2018)',
    xaxis_title='', yaxis=dict(title='Faturamento (R$)'),
    yaxis2=dict(title='Pedidos', overlaying='y', side='right', showgrid=False),
    legend=dict(orientation='h', y=1.12, x=0), hovermode='x unified',
    template='plotly_white', height=460,
)
fig.show()

## 2. Top 10 categorias de produto

Das categorias com faturamento no período, as 10 maiores concentram a maior parte do resultado —
poucas categorias sustentam o negócio.

In [48]:
cat_total = itens.groupby('product_category_name_pt').agg(
    faturamento=('price', 'sum'),
    quantidade=('order_item_id', 'count'),
).sort_values('faturamento', ascending=False)

top10 = cat_total.head(10)
pct_faturamento = 100 * top10['faturamento'].sum() / cat_total['faturamento'].sum()
pct_quantidade  = 100 * top10['quantidade'].sum() / cat_total['quantidade'].sum()

tabela_top10 = top10.reset_index().round(2)
tabela_top10['pct_faturamento'] = (100 * top10['faturamento'] / cat_total['faturamento'].sum()).round(1).values
tabela_top10['pct_quantidade']  = (100 * top10['quantidade'] / cat_total['quantidade'].sum()).round(1).values
display(tabela_top10)

resumo_top10 = pd.DataFrame([{
    'categorias_distintas':   cat_total.shape[0],
    'pct_faturamento_top10':  round(pct_faturamento, 1),
    'pct_quantidade_top10':   round(pct_quantidade, 1),
}])
display(resumo_top10)

,product_category_name_pt,faturamento,quantidade,pct_faturamento,pct_quantidade
0,Saúde & Beleza,1253993.86,9619,9.4,8.7
1,Relógios & Presentes,1201645.44,5986,9.0,5.4
2,"Cama, Mesa & Banho",1036509.69,11107,7.8,10.0
3,Esporte & Lazer,984715.33,8622,7.4,7.8
4,Computador e Acessórios,910555.00,7806,6.8,7.1
5,Móveis & Decoração,723881.71,8265,5.4,7.5
6,Hobbies,634179.85,3788,4.7,3.4
7,Artigos para Casa,630961.59,6952,4.7,6.3
8,Acessórios para Carro,590886.86,4223,4.4,3.8
9,Jardinagem,483896.58,4342,3.6,3.9


,categorias_distintas,pct_faturamento_top10,pct_quantidade_top10
0,73,63.2,63.9


In [49]:
## Top 10 Categorias

fig = make_subplots(
    rows=2, cols=2, column_widths=[0.68, 0.32], row_heights=[0.5, 0.5],
    specs=[[{'secondary_y': True, 'rowspan': 2}, {'type': 'domain'}],
           [None, {'type': 'domain'}]],
    horizontal_spacing=0.08, vertical_spacing=0.12,
)
fig.add_trace(go.Bar(
    x=top10.index, y=top10['faturamento'], name='Faturamento (R$)', marker_color='#2a78d6',
), row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(
    x=top10.index, y=top10['quantidade'], name='Qtd. itens vendidos', mode='lines+markers',
    line=dict(color='#eb6834', width=2), marker=dict(size=7),
), row=1, col=1, secondary_y=True)
fig.add_trace(go.Indicator(
    mode='gauge+number', value=pct_faturamento, number={'suffix': '%'},
    title={'text': 'Peso do Top 10<br>no Faturamento'},
    gauge={'axis': {'range': [0, 100]}, 'bar': {'color': '#2a78d6'}},
), row=1, col=2)
fig.add_trace(go.Indicator(
    mode='gauge+number', value=pct_quantidade, number={'suffix': '%'},
    title={'text': 'Peso do Top 10<br>no Volume'},
    gauge={'axis': {'range': [0, 100]}, 'bar': {'color': '#eb6834'}},
), row=2, col=2)

fig.update_yaxes(title_text='Faturamento (R$)', row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text='Qtd. de itens', row=1, col=1, secondary_y=True, showgrid=False)
fig.update_layout(
    title=(f'Top 10 categorias — {pct_faturamento:.0f}% do faturamento e {pct_quantidade:.0f}% do '
           f'volume, de {cat_total.shape[0]} categorias distintas'),
    template='plotly_white', height=560, legend=dict(orientation='h', y=1.1, x=0),
)
fig.show()

## 3. Histórico de faturamento — Top 5 categorias

Entre as 5 categorias de maior faturamento, 4 caem a partir de meados de 2018. A exceção é
**Saúde & Beleza**, que segue em crescimento contínuo — e é justamente a que mais pesa no faturamento
total, o que explica por que o platô do total (gráfico 1) não é uma queda mais acentuada.

In [50]:
top5_mensal = (
    itens[itens['product_category_name_english'].isin(TOP5)]
    .groupby(['ano_mes', 'product_category_name_english'])['price'].sum()
    .unstack()[TOP5]
)

NOMES_TOP5 = {
    cat: df_products.loc[df_products['product_category_name_english'] == cat, 'product_category_name_pt'].iloc[0]
    for cat in TOP5
}

tabela_top5_faturamento = top5_mensal.rename(columns=NOMES_TOP5).round(2).rename_axis(columns=None).reset_index()
display(tabela_top5_faturamento)

,ano_mes,Saúde & Beleza,Relógios & Presentes,"Cama, Mesa & Banho",Esporte & Lazer,Computador e Acessórios
0,2017-01,12561.32,8086.52,3960.16,9717.62,3924.14
1,2017-02,22838.79,11756.21,16282.73,16372.16,11972.59
2,2017-03,25995.25,26770.38,25773.02,27001.59,28624.60
3,2017-04,22935.75,23487.78,24347.69,24864.33,20691.06
4,2017-05,46786.02,37973.90,33346.45,36163.50,45634.78
5,2017-06,32029.39,28948.63,35114.81,32898.33,37007.08
6,2017-07,34896.86,36804.56,63888.75,38813.41,38709.00
7,2017-08,49873.90,36419.20,57137.23,41732.26,35025.72
8,2017-09,51537.65,47135.60,52473.20,50167.48,28930.98
9,2017-10,41915.72,65959.53,46198.00,49751.45,44022.09


In [51]:
# Faturamento Mensal por Categoria

fig = go.Figure()
for cat in TOP5:
    fig.add_trace(go.Scatter(
        x=top5_mensal.index, y=top5_mensal[cat], name=NOMES_TOP5[cat],
        mode='lines+markers', line=dict(color=CORES_CATEGORIA[cat], width=2), marker=dict(size=6),
    ))
fig.add_vrect(x0='2018-01', x1='2018-08', fillcolor='#e34948', opacity=0.06, line_width=0)
fig.update_layout(
    title='Faturamento mensal por categoria — Top 5 (jan/2017–ago/2018)',
    xaxis_title='', yaxis_title='Faturamento (R$)',
    template='plotly_white', height=460, hovermode='x unified',
    legend=dict(orientation='h', y=1.15, x=0),
)
fig.show()

## 4. Variação de preço médio — Top 5 categorias

Hipótese testada e **descartada**: reajuste de preço não explica a queda. A variação % mês a mês do
preço médio é ruído — sem tendência de alta sustentada em nenhuma das 5 categorias no período em que
o faturamento cai.

In [52]:
preco_medio = (
    itens[itens['product_category_name_english'].isin(TOP5)]
    .groupby(['ano_mes', 'product_category_name_english'])['price'].mean()
    .unstack()[TOP5]
)
variacao_preco = preco_medio.pct_change() * 100

tabela_variacao_preco = variacao_preco.rename(columns=NOMES_TOP5).round(1).rename_axis(columns=None).reset_index()
display(tabela_variacao_preco)

,ano_mes,Saúde & Beleza,Relógios & Presentes,"Cama, Mesa & Banho",Esporte & Lazer,Computador e Acessórios
0,2017-01,NaN,NaN,NaN,NaN,NaN
1,2017-02,-6.9,-56.8,19.6,-13.6,-10.8
2,2017-03,-10.5,4.0,-9.6,29.3,45.6
3,2017-04,-1.5,-26.7,-0.4,-15.9,-7.5
4,2017-05,32.9,16.2,1.7,0.3,-6.3
5,2017-06,-23.6,-17.0,1.7,3.8,-0.6
6,2017-07,-10.4,-6.2,20.0,-15.6,-16.3
7,2017-08,25.5,-21.6,1.1,-2.1,-15.7
8,2017-09,-2.1,2.4,-11.8,10.8,14.3
9,2017-10,-14.4,19.4,-13.9,-2.9,19.6


In [53]:
# Grafico de Variação do Mês

fig = go.Figure()
for cat in TOP5:
    fig.add_trace(go.Scatter(
        x=variacao_preco.index, y=variacao_preco[cat], name=NOMES_TOP5[cat],
        mode='lines+markers', line=dict(color=CORES_CATEGORIA[cat], width=2), marker=dict(size=6),
    ))
fig.add_hline(y=0, line_color='#8a8a86', line_width=1)
fig.update_layout(
    title='Variação % mês a mês do preço médio — Top 5 categorias',
    xaxis_title='', yaxis_title='Variação do preço médio (%)',
    template='plotly_white', height=440, hovermode='x unified',
    legend=dict(orientation='h', y=1.15, x=0),
)
fig.show()

In [54]:
# Quantidade de produtos cadastrados por categoria (df_products = cadastro, nao filtrado por data)
qtd_produtos = (
    df_products[df_products['product_category_name_english'].isin(TOP5)]
    .groupby('product_category_name_pt')['product_id'].nunique()
    .reindex([df_products.loc[df_products['product_category_name_english'] == c,
                               'product_category_name_pt'].iloc[0] for c in TOP5])
    .rename('produtos_cadastrados').reset_index()
)
display(qtd_produtos)

,product_category_name_pt,produtos_cadastrados
0,Saúde & Beleza,2444
1,Relógios & Presentes,1329
2,"Cama, Mesa & Banho",3029
3,Esporte & Lazer,2867
4,Computador e Acessórios,1639


## 5. Frete — sinal comum entre categorias (jun–jul/2018)

Preço e volume de vendas oscilam sem padrão comum entre categorias. O **frete** é diferente — e ao
comparar o custo médio de frete com o preço médio e o volume de itens vendidos (padronizados em
Z-score, para comparar séries de escalas diferentes), o pico aparece **na mesma janela em todas as
categorias**: não é maio, o mês da greve (21–30/mai/2018) — é **junho (e julho)**.

**Por que junho e não maio:** o custo de frete só entra na base quando o pedido é efetivamente
despachado. A greve bloqueou estradas no fim de maio e os fretes
recontratados a preços mais altos no pós-greve — só aparecem na fatura de pedidos despachados a
partir de junho. Maio, em si, ainda carrega majoritariamente frete contratado antes do bloqueio, e
por isso aparece **baixo** na série, não alto.

> **Ressalva:** o platô do faturamento total já começa em jan/2018, 4 meses antes da greve. A greve
> explica um choque agudo em jun–jul, não a causa raiz do platô.

In [55]:
geral_frete = itens.groupby('ano_mes').agg(
    preco=('price', 'mean'), frete=('freight_value', 'mean'), volume=('order_item_id', 'count'),
)
variacao_geral = geral_frete.pct_change() * 100
zscore_geral = (variacao_geral - variacao_geral.mean()) / variacao_geral.std()

tabela_frete_geral = zscore_geral.round(2).rename_axis('ano_mes').reset_index()
tabela_frete_geral.columns = ['ano_mes', 'zscore_preco', 'zscore_frete', 'zscore_volume']
display(tabela_frete_geral)

,ano_mes,zscore_preco,zscore_frete,zscore_volume
0,2017-01,NaN,NaN,NaN
1,2017-02,0.14,1.87,2.72
2,2017-03,-0.24,-0.73,1.18
3,2017-04,1.32,0.11,-0.78
4,2017-05,-1.48,-0.30,1.19
5,2017-06,-0.18,-0.04,-0.87
6,2017-07,-1.49,-0.37,0.34
7,2017-08,1.08,-0.19,-0.20
8,2017-09,1.85,0.40,-0.51
9,2017-10,-0.56,-0.25,-0.15


In [56]:
CORES_METRICA = {'preco': '#2a78d6', 'volume': '#1baf7a', 'frete': '#e34948'}
NOMES_METRICA = {'preco': 'Preço', 'volume': 'Volume', 'frete': 'Frete'}

fig = go.Figure()
for metrica in ['preco', 'volume', 'frete']:
    fig.add_trace(go.Scatter(
        x=zscore_geral.index, y=zscore_geral[metrica], name=NOMES_METRICA[metrica],
        mode='lines+markers', line=dict(color=CORES_METRICA[metrica], width=2), marker=dict(size=6),
    ))
fig.add_vrect(x0='2018-06', x1='2018-07', fillcolor='#e34948', opacity=0.10, line_width=0,
              annotation_text='jun–jul/2018', annotation_position='top left')
fig.add_annotation(x='2018-05', y=1.0, yref='paper', yanchor='bottom', showarrow=False,
                    text='greve (mai/2018)', font=dict(color='#4a3aa7', size=11))
fig.add_vline(x='2018-05', line_color='#4a3aa7', line_width=1, line_dash='dot')
fig.update_layout(
    title='Variação padronizada (Z-score) de preço, volume e frete — Total (jan/2017–ago/2018)',
    xaxis_title='', yaxis_title='Z-score da variação mensal',
    template='plotly_white', height=460, hovermode='x unified',
    legend=dict(orientation='h', y=1.15, x=0),
)
fig.show()

### 5.1 Por categoria — mesmo padrão nas Top 4 (todas exceto Saúde & Beleza)

O pico de frete no total não é puxado por uma categoria isolada. Quebrando por categoria — nas 4 do
Top 5 que estão em queda (Saúde & Beleza fica de fora, é a única em crescimento) — o mesmo padrão se
repete em cada uma, reforçando que é um choque comum, não um problema pontual de uma categoria.

In [57]:
CATS_FRETE = [c for c in TOP5 if c != 'health_beauty']  # a unica em crescimento fica de fora

g = (
    itens[itens['product_category_name_english'].isin(CATS_FRETE)]
    .groupby(['ano_mes', 'product_category_name_english'])
    .agg(preco=('price', 'mean'), frete=('freight_value', 'mean'), volume=('order_item_id', 'count'))
)

zscores = {}
for cat in CATS_FRETE:
    serie = g.xs(cat, level=1)
    variacao = serie.pct_change() * 100
    zscores[cat] = (variacao - variacao.mean()) / variacao.std()

tabela_frete_zscore = pd.concat({NOMES_TOP5[cat]: z for cat, z in zscores.items()}, axis=1).round(2)
tabela_frete_zscore.columns = [f'{cat} - {metrica.capitalize()}' for cat, metrica in tabela_frete_zscore.columns]
display(tabela_frete_zscore.reset_index())

,ano_mes,Relógios & Presentes - Preco,Relógios & Presentes - Frete,Relógios & Presentes - Volume,"Cama, Mesa & Banho - Preco","Cama, Mesa & Banho - Frete","Cama, Mesa & Banho - Volume",Esporte & Lazer - Preco,Esporte & Lazer - Frete,Esporte & Lazer - Volume,Computador e Acessórios - Preco,Computador e Acessórios - Frete,Computador e Acessórios - Volume
0,2017-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-02,-2.84,-1.39,3.47,2.06,0.17,3.52,-1.10,-1.10,2.91,-0.70,1.48,2.87
2,2017-03,0.51,-0.27,1.50,-1.17,-1.34,0.81,2.35,1.33,0.52,3.15,1.18,0.47
3,2017-04,-1.18,0.56,-0.17,-0.15,-0.06,-0.48,-1.29,-1.02,-0.12,-0.48,-0.43,-0.70
4,2017-05,1.18,0.28,0.16,0.08,0.02,0.16,0.02,0.11,1.14,-0.39,-0.57,1.43
5,2017-06,-0.65,-0.80,-0.64,0.08,-0.21,-0.34,0.30,1.42,-0.89,-0.00,-0.22,-0.65
6,2017-07,-0.05,0.32,0.10,2.10,1.02,0.43,-1.26,-1.31,0.95,-1.07,0.06,-0.06
7,2017-08,-0.90,-0.17,-0.06,0.02,-0.72,-0.58,-0.18,-0.48,-0.11,-1.04,-0.95,-0.30
8,2017-09,0.42,-0.02,-0.06,-1.41,0.09,-0.33,0.86,0.40,-0.15,1.01,1.92,-0.78
9,2017-10,1.36,-0.11,-0.21,-1.64,-1.21,-0.36,-0.25,-0.71,-0.38,1.37,-0.30,-0.03


In [58]:
fig = make_subplots(rows=2, cols=2, subplot_titles=[NOMES_TOP5[c] for c in CATS_FRETE])

for i, cat in enumerate(CATS_FRETE):
    row, col = divmod(i, 2)
    z = zscores[cat]
    for metrica, nome in [('preco', 'Preço'), ('volume', 'Volume'), ('frete', 'Frete')]:
        fig.add_trace(go.Scatter(
            x=z.index, y=z[metrica], name=nome, mode='lines+markers',
            line=dict(color=CORES_METRICA[metrica], width=2), marker=dict(size=4),
            showlegend=(i == 0), legendgroup=metrica,
        ), row=row + 1, col=col + 1)
    fig.add_vrect(x0='2018-06', x1='2018-07', fillcolor='#e34948', opacity=0.10, line_width=0,
                  row=row + 1, col=col + 1)

fig.update_layout(
    title='Variação padronizada (Z-score) de preço, volume e frete — Top 4 categorias em queda',
    template='plotly_white', height=680, legend=dict(orientation='h', y=1.08, x=0.3),
)
fig.show()

## 6. Vendedores — distribuição mensal (novos / ativos recorrentes / inativos)

Para cada mês, um vendedor é classificado como: **novo** (primeira venda naquele mês), **ativo
recorrente** (já vendia antes e vendeu de novo neste mês) ou **inativo** (já vendia antes, mas não
vendeu neste mês — cadastro "elegível" que ficou parado). A pilha cresce mês a mês porque a base
elegível (todo vendedor que já vendeu alguma vez) só aumenta, nunca diminui.

> **Cuidado, ver seção 6.1 logo abaixo:** essa é uma métrica **cumulativa**. O crescimento da fatia
> "inativos" pode ser puro efeito matemático de acúmulo (quanto mais meses passam, maior a chance
> acumulada de um vendedor antigo não vender *naquele* mês específico), não necessariamente piora
> real na retenção. Não tirar conclusão deste gráfico sozinho.

In [59]:
primeira_venda    = itens.groupby('seller_id')['ano_mes'].min()
vendedores_no_mes = itens.groupby(['seller_id', 'ano_mes']).size().unstack(fill_value=0) > 0

meses_recorte = sorted(itens['ano_mes'].unique())

linhas = []
for m in meses_recorte:
    elegiveis = primeira_venda[primeira_venda <= m].index
    novos     = int((primeira_venda == m).sum())
    antigos   = [s for s in elegiveis if primeira_venda[s] < m]
    ativos_recorrentes = sum(1 for s in antigos if vendedores_no_mes.loc[s, m])
    inativos = len(antigos) - ativos_recorrentes
    linhas.append((m, novos, ativos_recorrentes, inativos))

vendedores_mensal = pd.DataFrame(
    linhas, columns=['ano_mes', 'novos', 'ativos_recorrentes', 'inativos']
)
vendedores_mensal['total'] = vendedores_mensal[['novos', 'ativos_recorrentes', 'inativos']].sum(axis=1)
vendedores_mensal['pct_inativos'] = (100 * vendedores_mensal['inativos'] / vendedores_mensal['total']).round(1)

display(vendedores_mensal)

,ano_mes,novos,ativos_recorrentes,inativos,total,pct_inativos
0,2017-01,227,0,0,227,0.0
1,2017-02,255,172,55,482,11.4
2,2017-03,176,323,159,658,24.2
3,2017-04,121,385,273,779,35.0
4,2017-05,126,457,322,905,35.6
5,2017-06,76,463,442,981,45.1
6,2017-07,117,489,492,1098,44.8
7,2017-08,129,579,519,1227,42.3
8,2017-09,128,603,624,1355,46.1
9,2017-10,148,628,727,1503,48.4


In [60]:
CORES_VENDEDOR = {'novos': '#2a78d6', 'ativos_recorrentes': '#1baf7a', 'inativos': '#eda100'}
NOMES_VENDEDOR = {'novos': 'Novos vendedores', 'ativos_recorrentes': 'Vendedores ativos recorrentes',
                  'inativos': 'Vendedores inativos'}

fig = go.Figure()
for col in ['novos', 'ativos_recorrentes', 'inativos']:
    fig.add_trace(go.Bar(
        x=vendedores_mensal['ano_mes'], y=vendedores_mensal[col], name=NOMES_VENDEDOR[col],
        marker_color=CORES_VENDEDOR[col],
    ))
fig.update_layout(
    barmode='stack',
    title='Distribuição mensal de vendedores: novos, ativos recorrentes e inativos',
    xaxis_title='', yaxis_title='Número de vendedores',
    template='plotly_white', height=480, legend=dict(orientation='h', y=1.12, x=0),
)
fig.show()

### 6.1 Validação — métricas não cumulativas

O gráfico acima, sozinho, sugeriria "a base de vendedores está se deteriorando". Duas métricas
**não cumulativas** testam essa leitura: 1 quantos vendedores vendem em cada mês, sem herdar
histórico; 2 entre quem vendeu no mês anterior, qual % vende de novo neste mês (retenção real, mês a mês) — em vez de comparar contra todo o histórico acumulado.

In [61]:
vendedores_por_mes_set = {m: set(itens.loc[itens['ano_mes'] == m, 'seller_id']) for m in meses_recorte}

ativos_no_mes = pd.Series({m: len(s) for m, s in vendedores_por_mes_set.items()}, name='vendedores_ativos')

retencao = []
for i in range(1, len(meses_recorte)):
    mes_anterior, mes_atual = meses_recorte[i - 1], meses_recorte[i]
    base = vendedores_por_mes_set[mes_anterior]
    retidos = base & vendedores_por_mes_set[mes_atual]
    retencao.append((mes_atual, len(base), len(retidos), round(100 * len(retidos) / len(base), 1)))

tabela_retencao = pd.DataFrame(
    retencao, columns=['ano_mes', 'ativos_mes_anterior', 'retidos_neste_mes', 'taxa_retencao_pct']
).merge(ativos_no_mes.rename('vendedores_ativos_no_mes'), left_on='ano_mes', right_index=True)

display(tabela_retencao)
print(f"\nMédia da taxa de retenção mês a mês: {tabela_retencao['taxa_retencao_pct'].mean():.1f}%")
print(f"Vendedores ativos: {ativos_no_mes.iloc[0]} em {meses_recorte[0]} -> "
      f"{ativos_no_mes.iloc[-1]} em {meses_recorte[-1]}")

,ano_mes,ativos_mes_anterior,retidos_neste_mes,taxa_retencao_pct,vendedores_ativos_no_mes
0,2017-02,227,172,75.8,427
1,2017-03,427,298,69.8,499
2,2017-04,499,349,69.9,506
3,2017-05,506,379,74.9,583
4,2017-06,583,390,66.9,539
5,2017-07,539,396,73.5,606
6,2017-08,606,477,78.7,708
7,2017-09,708,515,72.7,731
8,2017-10,731,532,72.8,776
9,2017-11,776,611,78.7,965



Média da taxa de retenção mês a mês: 73.5%
Vendedores ativos: 227 em 2017-01 -> 1278 em 2018-08


In [62]:
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(
    x=list(ativos_no_mes.index), y=ativos_no_mes.values, name='Vendedores ativos no mês',
    marker_color='#2a78d6',
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=tabela_retencao['ano_mes'], y=tabela_retencao['taxa_retencao_pct'], name='Retenção mês a mês (%)',
    mode='lines+markers', line=dict(color='#e34948', width=2), marker=dict(size=7),
), secondary_y=True)
fig.update_yaxes(title_text='Vendedores ativos no mês', secondary_y=False)
fig.update_yaxes(title_text='Retenção mês a mês (%)', range=[0, 100], secondary_y=True, showgrid=False)
fig.update_layout(
    title='Vendedores ativos por mês (não cumulativo) e retenção real mês a mês',
    xaxis_title='', template='plotly_white', height=460,
    legend=dict(orientation='h', y=1.12, x=0), hovermode='x unified',
)
fig.show()

**Conclusão da validação:** o gráfico da seção 6 é um **artefato matemático**, não um sinal de piora.
Vendedores ativos por mês (métrica sem acúmulo) sobem o período inteiro — de 227 em jan/2017 a 1.278
em ago/2018 — inclusive durante o platô de faturamento. A retenção real mês a mês oscila entre ~67% e
~79%, sem tendência de queda; a média de 2018 (73–77%) é, se algo, ligeiramente **melhor** que a de
2017. Não há evidência de que a base de vendedores esteja se deteriorando.

**Como isso entra no relatório:** esse achado **reforça** a conclusão já estabelecida — o platô não
vem de um problema operacional interno (nem preço, nem logística estrutural, nem, agora, a base de
vendedores). Ele é mais uma evidência a favor de investigar o lado da demanda/cliente, não uma peça
nova da causa. **Não incluir a versão cumulativa (seção 6) na apresentação isoladamente** — se for
usar o tema vendedores no deck, usar o gráfico não cumulativo (6.1), que é o correto.

## 7. Atraso de entrega e satisfação — o sinal que já vinha antes da greve

O gráfico 5 mostrou o choque de **custo** de frete em maio–jun/2018 (greve). Antes disso, já havia um
sinal diferente: o **prazo** de entrega (`atraso_dias`, calculado no notebook 01) piora de forma aguda
em fev–mar/2018 — dois a três meses *antes* da greve — e a nota de satisfação (`review_score_medio`)
cai junto.

Primeiro teste: **é excesso de demanda?** Comparado a jan e abr/2018 (pedidos parecidos), fev e mar não
têm pico de volume que justifique a sobrecarga — o teste **descarta volume como causa**, como já foi
feito com preço (gráfico 4).

In [63]:
geral_qualidade = df_combined.groupby('ano_mes').agg(
    pedidos=('order_id', 'nunique'),
    entregues=('entrega_atrasada', 'count'),
    atrasados=('entrega_atrasada', 'sum'),
    review_medio=('review_score_medio', 'mean'),
).reset_index()
geral_qualidade['pct_atraso']   = (100 * geral_qualidade['atrasados'] / geral_qualidade['entregues']).round(1)
geral_qualidade['review_medio'] = geral_qualidade['review_medio'].round(3)

tabela_qualidade = geral_qualidade[['ano_mes', 'pedidos', 'pct_atraso', 'review_medio']]
display(tabela_qualidade)

print('Teste de volume (jan a abr/2018): pedidos não têm pico em fev-mar que justifique o salto de atraso —')
print(tabela_qualidade[tabela_qualidade['ano_mes'].between('2018-01', '2018-04')].to_string(index=False))

,ano_mes,pedidos,pct_atraso,review_medio
0,2017-01,800,2.933333,4.063
1,2017-02,1780,2.964307,4.016
2,2017-03,2682,4.556167,4.075
3,2017-04,2404,6.556665,4.046
4,2017-05,3700,2.990127,4.142
5,2017-06,3245,3.030303,4.148
6,2017-07,4026,2.789256,4.175
7,2017-08,4331,2.909611,4.237
8,2017-09,4285,4.385542,4.188
9,2017-10,4631,4.175971,4.125


Teste de volume (jan a abr/2018): pedidos não têm pico em fev-mar que justifique o salto de atraso —
ano_mes  pedidos pct_atraso  review_medio
2018-01     7269   5.700948         4.042
2018-02     6728  14.139719         3.827
2018-03     7211  18.963301         3.754
2018-04     6939   4.501324         4.163


In [64]:
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(
    x=geral_qualidade['ano_mes'], y=geral_qualidade['pct_atraso'], name='% pedidos atrasados',
    marker_color='#e34948',
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=geral_qualidade['ano_mes'], y=geral_qualidade['review_medio'], name='Nota média',
    mode='lines+markers', line=dict(color='#2a78d6', width=2), marker=dict(size=7),
), secondary_y=True)
fig.add_vrect(x0='2018-02', x1='2018-03', fillcolor='#e34948', opacity=0.10, line_width=0,
              annotation_text='fev–mar/2018', annotation_position='top left')
fig.add_vline(x='2018-05', line_color='#4a3aa7', line_width=1, line_dash='dot')
fig.add_annotation(x='2018-05', y=1.0, yref='paper', yanchor='bottom', showarrow=False,
                    text='greve (mai/2018)', font=dict(color='#4a3aa7', size=11))
fig.update_yaxes(title_text='% pedidos atrasados', secondary_y=False)
fig.update_yaxes(title_text='Nota média', range=[3.5, 4.4], secondary_y=True, showgrid=False)
fig.update_layout(
    title='Atraso de entrega e satisfação — visão geral (jan/2017–ago/2018)',
    xaxis_title='', template='plotly_white', height=460,
    legend=dict(orientation='h', y=1.12, x=0), hovermode='x unified',
)
fig.show()

### 7.1 É sistêmico — mesmo padrão nas Top 5 categorias (40% do faturamento)

Se o problema fosse de uma categoria específica, o sinal apareceria isolado. Testando nas 5 categorias que concentram 40% do faturamento e
39% do volume do período

In [65]:
itens_qualidade = itens.merge(
    df_combined[['order_id', 'entrega_atrasada', 'review_score_medio']], on='order_id', how='left'
)
itens_top5_qualidade = itens_qualidade[itens_qualidade['product_category_name_english'].isin(TOP5)]

atraso_top5 = (
    itens_top5_qualidade.groupby(['ano_mes', 'product_category_name_english'])['entrega_atrasada'].mean()
    .unstack()[TOP5] * 100
)

tabela_atraso_top5 = atraso_top5.rename(columns=NOMES_TOP5).round(1).rename_axis(columns=None).reset_index()
display(tabela_atraso_top5)

,ano_mes,Saúde & Beleza,Relógios & Presentes,"Cama, Mesa & Banho",Esporte & Lazer,Computador e Acessórios
0,2017-01,1.219512,9.090909,4.347826,4.054054,3.225806
1,2017-02,1.948052,0.0,1.910828,1.342282,5.0
2,2017-03,4.326923,2.531646,4.659498,3.157895,6.432749
3,2017-04,5.0,5.263158,4.059041,5.418719,4.545455
4,2017-05,3.169014,1.550388,4.432133,2.960526,2.588997
5,2017-06,4.330709,2.521008,2.168022,4.460967,1.968504
6,2017-07,9.415584,3.08642,1.054482,2.133333,4.0625
7,2017-08,3.977273,2.369668,2.766798,5.035971,2.346041
8,2017-09,7.180851,3.787879,3.219697,2.207506,2.834008
9,2017-10,3.428571,4.220779,3.690037,2.813853,3.559871


In [66]:
fig = go.Figure()
for cat in TOP5:
    fig.add_trace(go.Scatter(
        x=atraso_top5.index, y=atraso_top5[cat], name=NOMES_TOP5[cat],
        mode='lines+markers', line=dict(color=CORES_CATEGORIA[cat], width=2), marker=dict(size=6),
    ))
fig.add_vrect(x0='2018-02', x1='2018-03', fillcolor='#e34948', opacity=0.10, line_width=0,
              annotation_text='fev–mar/2018', annotation_position='top left')
fig.add_vline(x='2018-05', line_color='#4a3aa7', line_width=1, line_dash='dot')
fig.add_annotation(x='2018-05', y=1.0, yref='paper', yanchor='bottom', showarrow=False,
                    text='greve (mai/2018)', font=dict(color='#4a3aa7', size=11))
fig.update_layout(
    title='% de pedidos atrasados por categoria — Top 5 (jan/2017–ago/2018)',
    xaxis_title='', yaxis_title='% pedidos atrasados',
    template='plotly_white', height=460, hovermode='x unified',
    legend=dict(orientation='h', y=1.15, x=0),
)
fig.show()

In [67]:
review_top5 = (
    itens_top5_qualidade.groupby(['ano_mes', 'product_category_name_english'])['review_score_medio'].mean()
    .unstack()[TOP5]
)

tabela_review_top5 = review_top5.rename(columns=NOMES_TOP5).round(2).rename_axis(columns=None).reset_index()
display(tabela_review_top5)

,ano_mes,Saúde & Beleza,Relógios & Presentes,"Cama, Mesa & Banho",Esporte & Lazer,Computador e Acessórios
0,2017-01,4.05,4.00,4.10,3.96,4.32
1,2017-02,4.07,4.00,4.20,4.14,3.93
2,2017-03,4.28,4.14,3.93,4.17,3.86
3,2017-04,4.00,3.95,4.02,4.03,3.83
4,2017-05,4.16,4.24,4.13,4.14,4.26
5,2017-06,4.23,4.14,4.01,4.37,3.97
6,2017-07,3.91,4.01,4.07,4.19,4.11
7,2017-08,4.26,4.37,4.07,4.33,4.24
8,2017-09,4.22,4.26,4.02,4.22,3.96
9,2017-10,4.20,4.01,3.85,4.14,4.04


In [68]:
fig = go.Figure()
for cat in TOP5:
    fig.add_trace(go.Scatter(
        x=review_top5.index, y=review_top5[cat], name=NOMES_TOP5[cat],
        mode='lines+markers', line=dict(color=CORES_CATEGORIA[cat], width=2), marker=dict(size=6),
    ))
fig.add_vrect(x0='2018-02', x1='2018-03', fillcolor='#e34948', opacity=0.10, line_width=0,
              annotation_text='fev–mar/2018', annotation_position='top left')
fig.add_vline(x='2018-05', line_color='#4a3aa7', line_width=1, line_dash='dot')
fig.add_annotation(x='2018-05', y=1.0, yref='paper', yanchor='bottom', showarrow=False,
                    text='greve (mai/2018)', font=dict(color='#4a3aa7', size=11))
fig.update_layout(
    title='Nota média de satisfação por categoria — Top 5 (jan/2017–ago/2018)',
    xaxis_title='', yaxis_title='Nota média',
    template='plotly_white', height=460, hovermode='x unified',
    legend=dict(orientation='h', y=1.15, x=0),
)
fig.show()

**Causa: não identificada com confiança.** Testamos a hipótese de reajuste do diesel pela Petrobras
e ela **não se sustenta**: o reajuste foi uma tendência gradual desde 2017 (política de paridade
internacional), sem salto específico em fev/2018, e não há registro de protesto, bloqueio ou
paralisação de caminhoneiros nesse período — o primeiro aviso formal da categoria foi só em
16–18/mai/2018, dias antes da greve em si. Uma pressão de custo gradual não explica um pico **agudo,
de 2 meses, que volta ao normal em abril** — esse formato é mais compatível com um evento pontual do
que com uma tendência econômica lenta.

Testamos também o Carnaval (10–14/fev/2018): explica parte de fevereiro, mas não sustenta a
magnitude nem a duração até março.

**Tratamos como um achado real com causa em aberto**, não como parte do mesmo evento da greve — são
dois sinais de estresse logístico em momentos diferentes de 2018, e só o segundo (Maio) tem causa
identificada.

## Conclusão

**O que foi descartado como causa interna:**
- **Preço** — variação mês a mês é ruído, sem tendência de alta nas 5 categorias que mais
  pesam no faturamento.
- **Base de vendedores** — vendedores ativos por mês crescem o período inteiro (227 →
  1.278) e a retenção real fica estável (~67–79%). Não há deterioração operacional do lado do seller.
- **Volume de pedidos** — não há pico de demanda em fev–mar/2018 que justifique, sozinho,
  o salto no atraso de entrega. Testamos também reajuste de diesel e Carnaval como explicação para o
  atraso — nenhum dos dois se sustenta com rigor (ver nota após o gráfico de satisfação).

**O que foi encontrado — e é a peça central da análise:**
O platô de faturamento começa em **jan/2018**. Dois sinais de estresse logístico aparecem depois,
**em momentos diferentes e por causas distintas**:

1. **Fev–mar/2018** — o prazo de entrega piora de forma aguda e sistêmica nas 5 categorias que
   respondem por 40% do faturamento e 39% do volume do período, e a satisfação cai junto. Volume, diesel e Carnaval foram testados e não explicam o padrão — **a causa raiz deste
   episódio fica em aberto**, é um achado real sem explicação fechada.
2. **Junho–julho/2018** — choque de **custo** de frete, com causa
   identificada: a greve dos caminhoneiros (21–30/mai/2018). O custo só aparece na fatura dos
   pedidos despachados depois do fim do bloqueio — por isso o pico é em jun–jul, não em maio.

Não afirmamos que são o mesmo evento — são dois episódios de estresse logístico que, juntos, marcam
justamente os meses em que o faturamento fica estagnado.

**Conclusão honesta:** o platô **não nasce de um problema interno da Olist** — preço e operação de
vendedores estão descartados, e a empresa segue captando novos vendedores e mantendo sua base
saudável. O que a atrapalhou a sustentar o ritmo de crescimento de 2017 foi uma sequência de choques
**logísticos externos** ao longo de 2018 — um com causa raiz ainda não identificada (fev–mar) e outro
com causa confirmada (a greve, refletida em jun–jul) — não uma falha de operação da própria Olist.

**Recomendações:**
1. **Curto prazo:** diversificar transportadoras e rotas para reduzir a exposição a um choque
   logístico único e concentrado — vale tanto para o episódio de causa desconhecida (fev–mar) quanto
   para replicar a greve.
2. **Médio prazo:** reforçar SLA e comunicação proativa de prazo ao cliente em períodos de risco
   logístico — a queda de satisfação em fev–mar mostra que o cliente sente o atraso diretamente,
   mesmo sem mudança de preço ou de vendedor.
3. **Investir em pesquisa de mercado do lado do cliente** (motivação de compra, comparação com
   concorrência) — com preço, vendedores e demanda descartados como causa interna, entender o cliente
   é o próximo passo natural para reacender o crescimento, não só reagir ao choque logístico.
4. **Investigar a causa raiz do episódio fev–mar/2018** — não identificamos o motivo com os dados
   disponíveis (não é volume, não é diesel, não é só Carnaval); vale checar internamente com a
   Olist/transportadoras o que aconteceu operacionalmente nesses dois meses.